In [0]:
spark


In [0]:
pip install aiohttp


In [0]:
from manager.nft_collection_manager import NFTCollectionManager

# 🔐 Get API key securely from Databricks Secrets
ALCHEMY_KEY = dbutils.secrets.get(
    scope="nft-scope",
    key="ALCHEMY_KEY"
)

# Initialize Manager (spark already exists in Databricks)
manager = NFTCollectionManager(spark, ALCHEMY_KEY)

# 🚀 Fetch Base NFTs (this internally calls repo + service)
manager.fetch_base("collection_1")

# 📊 Display stored base table
display(spark.table("nft_base_table"))

In [0]:
from repository.collection_repository import CollectionRepository

repo = CollectionRepository(spark)

df = spark.table(repo.table_name)

display(df.select("collection_name", "contract_address", "status"))

In [0]:
from services.nft_base_service import NFTBaseService
from repository.collection_repository import CollectionRepository

# 🔐 Securely fetch API key (do NOT hardcode)
ALCHEMY_KEY = dbutils.secrets.get(
    scope="nft-scope",
    key="ALCHEMY_KEY"
)

# Spark already exists in Databricks Serverless

# Initialize repository and service
repo = CollectionRepository(spark)
base_service = NFTBaseService(spark, ALCHEMY_KEY)

collection_name = "collection_1"

# Get contract address using modular function
contract_address = repo.get_collection_id(collection_name)

if not contract_address:
    raise ValueError(f"Collection '{collection_name}' not found")

print("Contract Address:", contract_address)

# 🚀 Fetch NFTs (this writes to nft_base_table)
base_service.fetch_nft_base(contract_address, collection_name)

# 📊 View stored table
display(spark.table("nft_base_table"))

In [0]:
display(spark.table("nft_base_table"))

In [0]:
# Spark table where metadata is stored
df = spark.table("nft_metadata_table")
df.display()

In [0]:
from pyspark.sql import SparkSession
from manager.nft_collection_manager import NFTCollectionManager
from repository.collection_repository import CollectionRepository

spark = SparkSession.builder.getOrCreate()

ALCHEMY_KEY = dbutils.secrets.get(
    scope="nft-scope",
    key="ALCHEMY_KEY"
)

manager = NFTCollectionManager(spark, ALCHEMY_KEY)

repo = CollectionRepository(spark)
collections = {
    "collection_1": "0xBd3531dA5CF5857e7CfAA92426877b022e612cf8"
}
repo.create_collections(collections)

manager.fetch_base("collection_1")
await manager.fetch_metadata("collection_1")